# Feature Engineering

## 1. Introduction

This notebook creates additional property, temporal, and geographic features for the California single-family residential AVM.

The notebook:

- reviews candidate variables that were not included in the baseline model;
- creates deterministic row-level features from existing property characteristics;
- adds property-age, room-ratio, area-ratio, logarithmic, amenity, and seasonal features;
- maps each property to a California Unified School District using latitude and longitude; and
- saves the engineered dataset before model-specific preprocessing.

## 2. Imports and Setup

In [2]:
from pathlib import Path
import geopandas as gpd
import joblib
import numpy as np
import pandas as pd

## 3. Load the Cleaned Dataset

The cleaned single-family residential dataset produced by the data cleaning notebook is loaded below.

In [3]:
df = pd.read_csv("../data/processed/crmls_sfr_cleaned_base.csv", low_memory=False)
print("Dataset shape:", df.shape)

Dataset shape: (398461, 70)


## 4. Review Candidate Features

The baseline model used 17 original property variables. This section reviews several additional variables that may provide useful geographic or property information.

17 original poperty variables:
target = "ClosePrice"

numeric_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "ParkingTotal",
    "Stories"
]

categorical_features = [
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "Levels",
    "PoolPrivateYN",
    "ViewYN",
    "AttachedGarageYN",
    "NewConstructionYN",
    "FireplaceYN"
]



### 4.1 Inspect Candidate Feature Missingness

In [4]:
candidate_features = [
    "Latitude",
    "Longitude",
    "City",
    "AssociationFee",
    "AssociationFeeFrequency",
    "HighSchoolDistrict",
    "CloseDate"
]

missing_df = (
    df[candidate_features]
    .isna()
    .mean()
    .to_frame(name="Missing_Percentage (%)")
    .multiply(100)
    .round(2)
)

print(missing_df)

                         Missing_Percentage (%)
Latitude                                   0.07
Longitude                                  0.07
City                                       0.13
AssociationFee                            30.58
AssociationFeeFrequency                   75.18
HighSchoolDistrict                        25.83
CloseDate                                  0.00


### 4.2 Inspect Candidate Feature Data Types

Date, numeric, and categorical fields must use consistent data types before feature engineering is performed.

In [5]:
dtype = df[candidate_features].dtypes
print(dtype)

Latitude                   float64
Longitude                  float64
City                        object
AssociationFee             float64
AssociationFeeFrequency     object
HighSchoolDistrict          object
CloseDate                   object
dtype: object


### 4.3 Candidate Feature Decisions

The candidate feature review supports the following decisions:

- `Latitude` and `Longitude` are retained because they provide precise property-location information and have low missingness.
- `City` is retained as an additional geographic category.
- `HighSchoolDistrict` is kept as a candidate variable, although approximately one quarter of its values are missing.
- `AssociationFee` is excluded from the first feature-engineering version because missing values and zero values cannot be interpreted consistently.
- `AssociationFeeFrequency` is excluded because most records are missing a payment frequency.
- `CloseDate` is not used directly as a model input, but it is used to create property-age, seasonal, and chronological split features.

**Section result:** The first engineered feature set will use the higher-quality property, geographic, and temporal variables while excluding the HOA fee variables.

## 5. Create Deterministic Property Features

The following features are created using values from each individual property record.

These transformations do not calculate statistics from other observations and therefore can be completed before the chronological split.

### 5.1 Prepare Date and Numeric Columns

A separate copy of the cleaned dataset is created. `CloseDate` is converted to datetime format, and the numeric fields required for feature engineering are converted to numeric values.

In [6]:
df_engineered = df.copy()


df_engineered["CloseDate"] = pd.to_datetime(
    df_engineered["CloseDate"],
    errors="coerce"
)


numeric_columns = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

for column in numeric_columns:
    df_engineered[column] = pd.to_numeric(
        df_engineered[column],
        errors="coerce"
    )

### 5.2 Create Property Age

`YearBuilt` identifies the construction year, but the age of the property at the valuation date is generally more meaningful.

For historical sales, property age is calculated as:

```text
PropertyAge = closing year - YearBuilt

In [7]:
df_engineered["PropertyAge"] = (
    df_engineered["CloseDate"].dt.year
    - df_engineered["YearBuilt"]
)

df_engineered.loc[
    df_engineered["PropertyAge"] < 0,
    "PropertyAge"
] = np.nan

### 5.3 Create Room and Area Ratios

Ratio features describe how the rooms and spaces of a property relate to one another.

A denominator must be greater than zero. If the denominator is zero, negative, or missing, the resulting ratio is left missing rather than creating an infinite or misleading value.

#### Bathroom-to-Bedroom Ratio

`BathBedRatio` represents the number of bathrooms available relative to the number of bedrooms.

In [8]:
df_engineered["BathBedRatio"] = (
    df_engineered["BathroomsTotalInteger"]
    / df_engineered["BedroomsTotal"].where(
        df_engineered["BedroomsTotal"] > 0
    )
)

#### Living Area per Bedroom

`LivingAreaPerBedroom` measures the amount of interior living space available per bedroom.

In [9]:
df_engineered["LivingAreaPerBedroom"] = (
    df_engineered["LivingArea"]
    / df_engineered["BedroomsTotal"].where(
        df_engineered["BedroomsTotal"] > 0
    )
)

#### Living Area per Bathroom

`LivingAreaPerBathroom` measures the amount of interior living space relative to the number of bathrooms.

In [10]:
df_engineered["LivingAreaPerBathroom"] = (
    df_engineered["LivingArea"]
    / df_engineered["BathroomsTotalInteger"].where(
        df_engineered["BathroomsTotalInteger"] > 0
    )
)

#### Lot-to-Living-Area Ratio

`LotToLivingRatio` compares the size of the lot with the size of the interior living area.

A larger value generally indicates that a greater portion of the property consists of outdoor land rather than interior space.

In [11]:
df_engineered["LotToLivingRatio"] = (
    df_engineered["LotSizeSquareFeet"]
    / df_engineered["LivingArea"].where(
        df_engineered["LivingArea"] > 0
    )
)

### 5.4 Create Log-Transformed Area Features

Living area and lot size are typically skewed because a small number of properties are substantially larger than most homes.

The natural-log transformation reduces the influence of very large values and may help a Linear Regression model represent nonlinear relationships more effectively.

In [12]:
df_engineered["LogLivingArea"] = np.log1p(
    df_engineered["LivingArea"].where(
        df_engineered["LivingArea"] > 0
    )
)

df_engineered["LogLotSize"] = np.log1p(
    df_engineered["LotSizeSquareFeet"].where(
        df_engineered["LotSizeSquareFeet"] > 0
    )
)

### 5.5 Create Amenity Summary Features

The original amenity variables record whether the property has a private pool, view, attached garage, new construction status, or fireplace.

Because Boolean values may appear in several formats, common true and false representations are first standardized to `1` and `0`.

Missing or unrecognized values remain missing rather than being interpreted as false.

In [13]:
amenity_columns = [
    "PoolPrivateYN",
    "ViewYN",
    "AttachedGarageYN",
    "NewConstructionYN",
    "FireplaceYN"
]

boolean_mapping = {
    True: 1,
    False: 0,
    "True": 1,
    "False": 0,
    "true": 1,
    "false": 0,
    "Yes": 1,
    "No": 0,
    "Y": 1,
    "N": 0,
    1: 1,
    0: 0
}

amenity_data = df_engineered[amenity_columns].replace(boolean_mapping)

amenity_data = amenity_data.apply(
    pd.to_numeric,
    errors="coerce"
)

C:\Users\Tangent\AppData\Local\Temp\ipykernel_3352\2359182500.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  amenity_data = df_engineered[amenity_columns].replace(boolean_mapping)


Two summary variables are created:

- `AmenityKnownCount`: the number of amenity fields with a known value.
- `AmenityCount`: the number of known amenities recorded as present.

Keeping both variables prevents a property with several unknown amenity values from being treated as equivalent to a property with confirmed false values.

In [14]:
# 1
df_engineered["AmenityKnownCount"] = (
    amenity_data.notna().sum(axis=1)
)

# 2
df_engineered["AmenityCount"] = (
    amenity_data.sum(
        axis=1,
        min_count=1
    )
)

### 5.6 Create Cyclical Month Features

Closing month may capture seasonal patterns in the real-estate market.

Month is cyclical rather than linear: December and January are adjacent in time even though their numeric values are 12 and 1. Sine and cosine transformations preserve this cyclical relationship.

The two resulting features are:

- `CloseMonthSin`
- `CloseMonthCos`

In a future prediction system, these features should be calculated from the valuation or *query date* rather than from an unknown future closing date.

In [15]:
close_month = df_engineered["CloseDate"].dt.month

df_engineered["CloseMonthSin"] = np.sin(
    2 * np.pi * close_month / 12
)

df_engineered["CloseMonthCos"] = np.cos(
    2 * np.pi * close_month / 12
)

### 5.6 Create Cyclical Month Features

Closing month may capture seasonal patterns in the real-estate market.

Month is cyclical rather than linear: December and January are adjacent in time even though their numeric values are 12 and 1. Sine and cosine transformations preserve this cyclical relationship.

The two resulting features are:

- `CloseMonthSin`
- `CloseMonthCos`

In a future prediction system, these features should be calculated from the valuation or *query date* rather than from an unknown future closing date.

In [16]:
df_engineered["split_month"] = (
    df_engineered["CloseDate"].dt.to_period("M")
)

### 5.8 Inspect the Engineered Features

The descriptive statistics and missing-value rates of the newly created features are reviewed below.

In [17]:
engineered_features = [
    "PropertyAge",
    "BathBedRatio",
    "LivingAreaPerBedroom",
    "LivingAreaPerBathroom",
    "LotToLivingRatio",
    "LogLivingArea",
    "LogLotSize",
    "AmenityCount",
    "AmenityKnownCount",
    "CloseMonthSin",
    "CloseMonthCos"
]

df_engineered[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
PropertyAge,398178.0,48.974408,27.402971,0.000000,27.000000,4.900000e+01,69.000000,2.490000e+02
BathBedRatio,398207.0,0.752952,0.286355,0.000000,0.666667,6.666667e-01,1.000000,8.750000e+01
LivingAreaPerBedroom,398281.0,580.333501,1234.889217,1.000000,445.333333,5.392500e+02,662.750000,7.695600e+05
LivingAreaPerBathroom,398262.0,793.927765,1839.277358,1.000000,650.000000,7.680000e+02,903.250000,1.154340e+06
LotToLivingRatio,391437.0,129.857072,7470.581953,0.000004,3.008215,4.251904e+00,6.014493,1.570930e+06
LogLivingArea,398461.0,7.519359,0.429529,1.098612,7.226209,7.496097e+00,7.791110,1.465219e+01
LogLotSize,391437.0,9.071218,0.947683,0.009950,8.641886,8.888619e+00,9.243388,2.145910e+01
AmenityCount,398440.0,2.186497,1.008600,0.000000,2.000000,2.000000e+00,3.000000,5.000000e+00
AmenityKnownCount,398461.0,4.580478,0.598003,0.000000,4.000000,5.000000e+00,5.000000,5.000000e+00
CloseMonthSin,398461.0,-0.009192,0.712625,-1.000000,-0.866025,-2.449294e-16,0.500000,1.000000e+00


In [18]:
missing_summary = (
    df_engineered[engineered_features]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .to_frame(name="Missing_Percentage (%)")
    .sort_values(by="Missing_Percentage (%)", ascending=False)
)

print(missing_summary)

                       Missing_Percentage (%)
LotToLivingRatio                         1.76
LogLotSize                               1.76
PropertyAge                              0.07
BathBedRatio                             0.06
LivingAreaPerBedroom                     0.05
LivingAreaPerBathroom                    0.05
AmenityCount                             0.01
LogLivingArea                            0.00
AmenityKnownCount                        0.00
CloseMonthSin                            0.00
CloseMonthCos                            0.00


## 6. Add Unified School District Geography

[The California School District Areas 2025–26 GeoJSON](https://data.ca.gov/dataset/california-school-district-areas-2025-26) is used to determine which Unified School District polygon contains each property.

The mapping is based only on latitude and longitude and does not use `ClosePrice` or any other target information.

### 6.1 Load the School District Boundary Data

The external GeoJSON file is loaded and its coordinate reference system, number of polygons, and district types are inspected.

In [19]:
DISTRICT_PATH = Path("../data/external/DistrictAreas2526.geojson")

districts = gpd.read_file(DISTRICT_PATH)

print("District shape:", districts.shape)
print("District CRS:", districts.crs)

print(districts["DistrictType"].value_counts(dropna=False))

District shape: (936, 51)
District CRS: EPSG:3857
DistrictType
Elementary    515
Unified       345
High           76
Name: count, dtype: int64


### 6.2 Keep Unified School Districts

Only records with `DistrictType == "Unified"` are retained, following the project feature-engineering requirement.

Elementary-only and high-school-only district polygons are not included in this version.

In [20]:
unified_districts = districts[districts["DistrictType"] == "Unified"][["DistrictName","CountyName", "geometry"]].copy()

print(
    "Unified district count:",
    len(unified_districts)
)

Unified district count: 345


In [21]:
unified_districts = (
    unified_districts
    .to_crs("EPSG:4326")
)

print(
    "New CRS:",
    unified_districts.crs
)

New CRS: EPSG:4326


### 6.3 Convert Property Coordinates to Geographic Points

Only records with both latitude and longitude are converted into geographic points.

Longitude is used as the x-coordinate and latitude is used as the y-coordinate. Records without a complete coordinate pair remain in the tabular dataset but cannot be assigned through the spatial join.

In [22]:
valid_coordinates = (
    df_engineered["Latitude"].notna()
    & df_engineered["Longitude"].notna()
)

In [23]:
property_points = gpd.GeoDataFrame(
    index=df_engineered.index[valid_coordinates],
    geometry=gpd.points_from_xy(
        df_engineered.loc[
            valid_coordinates,
            "Longitude"
        ],
        df_engineered.loc[
            valid_coordinates,
            "Latitude"
        ]
    ),
    crs="EPSG:4326"
)

print(
    "Property point count:",
    len(property_points)
)
property_points

Property point count: 398174


,geometry
0,POINT (-117.3406 33.14727)
1,POINT (-117.20227 34.23598)
2,POINT (-116.44728 33.78377)
3,POINT (-121.5864 39.76773)
4,POINT (-117.14204 32.97721)
...,...
398456,POINT (-118.18774 33.78687)
398457,POINT (-119.14586 34.37768)
398458,POINT (-118.45237 34.27526)
398459,POINT (-121.96159 37.85465)


### 6.4 Perform the Spatial Join

A spatial join identifies the Unified School District polygon that contains each property point.

The left join preserves all property points, including properties that do not fall within one of the retained Unified School District polygons.

In [24]:
district_match = gpd.sjoin(
    property_points,
    unified_districts[
        [
            "DistrictName",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

district_match.head()

,geometry,index_right,DistrictName
0,POINT (-117.3406 33.14727),582.0,Carlsbad Unified
1,POINT (-117.20227 34.23598),533.0,Rim of the World Unified
2,POINT (-116.44728 33.78377),477.0,Palm Springs Unified
3,POINT (-121.5864 39.76773),29.0,Paradise Unified
4,POINT (-117.14204 32.97721),568.0,Poway Unified


### 6.4 Add the Unified School District Feature

The matched district name is aligned back to the original dataframe index and stored as `UnifiedSchoolDistrict`.

In [25]:
df_engineered["UnifiedSchoolDistrict"] = (
    district_match["DistrictName"]
    .reindex(df_engineered.index)
)

### 6.7 Evaluate Spatial-Matching Coverage

The match rate is calculated among properties with valid coordinate pairs.

In [26]:
valid_coordinate_count = valid_coordinates.sum()

matched_count = (
    df_engineered.loc[
        valid_coordinates,
        "UnifiedSchoolDistrict"
    ]
    .notna()
    .sum()
)

match_rate = (
    matched_count
    / valid_coordinate_count
    * 100
)

print(
    "Valid coordinates:",
    valid_coordinate_count
)

print(
    "Matched Unified districts:",
    matched_count
)

print(
    "Match rate:",
    round(match_rate, 2),
    "%"
)

Valid coordinates: 398174
Matched Unified districts: 297989
Match rate: 74.84 %


In [27]:
df_engineered[
    "UnifiedSchoolDistrict"
].value_counts(
    dropna=False
).head(20)

UnifiedSchoolDistrict
NaN                            100472
Los Angeles Unified             38230
San Diego Unified               10559
Capistrano Unified               7089
Desert Sands Unified             7028
Palm Springs Unified             5980
Oakland Unified                  5170
Corona-Norco Unified             5008
Hemet Unified                    4868
Long Beach Unified               4699
Riverside Unified                4573
Temecula Valley Unified          4419
Mt. Diablo Unified               4155
San Bernardino City Unified      3710
Lake Elsinore Unified            3481
Saddleback Valley Unified        3341
Poway Unified                    3271
Orange Unified                   3065
Hesperia Unified                 3040
Newport-Mesa Unified             3037
Name: count, dtype: int64

## 7. Save the Engineered Dataset

The completed dataset is saved in Parquet format.

The saved file contains:

- the cleaned original variables;
- deterministic engineered property features;
- temporal features;
- `UnifiedSchoolDistrict`;
- the original target variable;
- the original missing values; and
- `split_month` for later chronological splitting.


In [28]:
output_path = (
    "../data/processed/"
    "crmls_sfr_engineered_full.parquet"
)

df_engineered.to_parquet(
    output_path,
    index=False
)

print(
    "Saved:",
    output_path
)

print(
    "Final shape:",
    df_engineered.shape
)

Saved: ../data/processed/crmls_sfr_engineered_full.parquet
Final shape: (398461, 83)
